# Elasticsearch Shards — Complete Guide

---

## What is a Shard?

### In plain terms (the library metaphor)

Imagine you have a massive library with millions of books. No single librarian can manage it all, so you split the library into sections. Each section is its own mini-library with its own catalog. When someone asks "find me all books about rockets", multiple librarians search their sections *at the same time* and bring back results.

**A shard is one of those sections** — a self-contained slice of your data that Elasticsearch can search independently and in parallel.

### In technical terms

A shard is a single **Lucene index instance**. When you create an index in Elasticsearch, it is divided into N primary shards (you set N upfront). Each shard holds a subset of your documents and has its own:

- Inverted index
- Stored segments
- Term dictionary (maps each term to which documents contain it)
- Term frequencies (used for relevance scoring)
- File handles on disk

Shard routing uses the formula:

```
shard = hash(doc_id) % num_primary_shards
```

This deterministically places and finds each document. You never have to think about it — Elasticsearch handles it automatically.

---

## Why is the Time Filter called "The Cheapest Filter"?

Yes — your instinct is exactly right. The time picker earns that name because of **shard pruning**.

Here is what happens step by step when you set a time range:

1. Your query arrives at the **coordinating node**
2. Elasticsearch checks the `@timestamp` min/max stored in each shard's metadata — *without opening a single document*
3. Any shard whose entire time range falls outside your window is **skipped entirely** — this is shard pruning
4. Only the surviving shards are searched, in parallel
5. Results are merged and returned

This happens at the **shard level**, before Elasticsearch reads any documents at all. A `WHERE` clause inside ES|QL, by contrast, runs *after* documents are loaded into the query engine — far more expensive.

The `_shards.skipped` field in the API response tells you exactly how many shards were pruned:

```json
"_shards": {
  "total": 10,
  "successful": 3,
  "skipped": 7,   ← these were never read
  "failed": 0
}
```

On a large cluster with time-partitioned indices, this number can be enormous — which is why the time picker has the biggest performance impact of all three Kibana filter layers.

---

## Why is a Shard called a "Lucene Index Instance"? Does Lucene have a Class?

Yes, literally. Here is the lineage:

**Apache Lucene** is an open-source Java search library that Elasticsearch is built on top of. It is the actual search engine doing the work under the hood. Elasticsearch adds the distributed system layer — networking, replication, REST API, clustering — on top of Lucene.

Inside Lucene's Java codebase there is a class called `IndexWriter` and a concept called a **Lucene index** — a directory of files on disk that stores the inverted index, term dictionary, and document store for a set of documents.

When Elasticsearch creates a shard, it is literally instantiating a Lucene index on that node's filesystem. That is why it is called an "instance" — each shard is one running copy of the Lucene index machinery, managing its own files independently.

```
Elasticsearch Index  (logical — what you name and query)
    └── Shard 0  (physical — a Lucene IndexWriter instance + files on disk)
    └── Shard 1
    └── Shard 2
        └── Segment 1  (immutable file written by Lucene)
        └── Segment 2
        └── Segment 3  ← Lucene merges small segments into larger ones over time
```

A few important Lucene internals worth knowing:

- Lucene writes data in **segments** — small immutable files. New documents create new segments.
- Lucene periodically **merges** smaller segments into larger ones to keep search fast (more segments = slower search, since Lucene searches them sequentially).
- Lucene uses **copy-on-write** for updates and deletes — it never modifies a document in place. It marks it deleted and writes a new one. Space is only reclaimed during a segment merge.
- Each Lucene index has a hard document limit of **2,147,483,519 documents** (2³¹ − 129). Sharding lets you exceed this across an Elasticsearch index.

---

## What Does "Shard" Mean in Other Contexts?

The word *shard* means a broken-off fragment — like a shard of glass or pottery. In computing it always carries this same meaning: a piece of something larger that was intentionally split up.

| Context | What is being sharded | Why |
|---|---|---|
| **Elasticsearch** | An index split into N Lucene instances | Parallel search, distributed storage |
| **Databases (PostgreSQL, MySQL, Vitess)** | A table or database split across multiple servers | Scale writes beyond one machine |
| **MongoDB** | A collection split across replica sets | Horizontal scaling |
| **Redis Cluster** | The keyspace split into 16,384 hash slots | Distribute memory across nodes |
| **Blockchain (Ethereum sharding)** | The chain split into parallel chains | Increase transaction throughput |
| **Game servers (MMOs)** | The player population split into parallel world instances | Handle millions of concurrent players |
| **DNS / CDNs** | Traffic split across geographic regions | Reduce latency, avoid single points of failure |

The pattern is always the same: one thing that is too big or too slow to handle in one place gets divided into independent pieces that can be managed in parallel. The pieces are called shards.

---

## Kibana Filter Layers — Recap

The three layers in Kibana's Discover/Lens interface each operate at a different depth in the stack:

| Layer | What it does | Where it runs | Cost |
|---|---|---|---|
| **Time picker** | Sets a time window | Shard metadata — before any document is read | Cheapest — shard pruning |
| **DSL filter bar** | Adds Elasticsearch Query DSL filters | Shard level — skips entire segments | Cheap — runs before ES\|QL |
| **ES\|QL WHERE** | Filters inside the query pipeline | Document level — after data is loaded | Most expensive |

Always apply the time picker first, DSL bar second, and push fine-grained logic into ES|QL last.

---

## Code Reference

### CLI — Inspect shards

```bash
# See all shards for an index
curl -X GET "http://localhost:9200/_cat/shards/logs-*?v&h=index,shard,prirep,state,docs,store,node"

# Shard-level stats
curl -X GET "http://localhost:9200/logs-*/_stats/docs,store?level=shards"

# Explain which shards a query hit
curl -X GET "http://localhost:9200/logs-*/_search?explain=true" \
  -H "Content-Type: application/json" \
  -d '{
    "query": {
      "range": {
        "@timestamp": {
          "gte": "now-7d",
          "lte": "now"
        }
      }
    }
  }'
```

### Python — elasticsearch client

```python
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

# 1. Inspect shard layout
shards = es.cat.shards(
    index="logs-*",
    v=True,
    h=["index", "shard", "prirep", "state", "docs", "store", "node"]
)
print(shards)

# 2. Query with time range (triggers shard pruning)
resp = es.search(
    index="logs-*",
    body={
        "query": {
            "bool": {
                "filter": [
                    {
                        "range": {
                            "@timestamp": {
                                "gte": "now-7d",
                                "lte": "now"
                            }
                        }
                    }
                ]
            }
        }
    }
)

print(f"Hits:            {resp['hits']['total']['value']}")
print(f"Shards searched: {resp['_shards']['successful']} / {resp['_shards']['total']}")
print(f"Shards skipped:  {resp['_shards'].get('skipped', 0)}")
# ^ This number tells you how much work shard pruning saved

# 3. ES|QL via HTTP
import requests

query = """
FROM logs-*
| WHERE @timestamp >= NOW() - 7 DAYS
| STATS errors = COUNT(*) BY DATE_TRUNC(1 DAYS, @timestamp)
| SORT @timestamp
"""

r = requests.post(
    "http://localhost:9200/_query",
    headers={"Content-Type": "application/json"},
    json={"query": query}
)
print(r.json())
```

---

## Key Numbers to Remember

| Limit / Rule of thumb | Value |
|---|---|
| Max documents per Lucene index (shard) | 2,147,483,519 |
| Optimal shard size | 10 – 50 GB |
| Max documents per shard (recommended) | 200 million |
| Default shards per index (ES 7.0+) | 1 primary, 1 replica |
| Max shards per node rule of thumb | 20 per GB of heap |
| Cluster default shard limit | 1,000 per non-frozen node |
